# PINN Generalization: Training on Single Agents, Predicting Combination

This notebook demonstrates a challenging generalization task for the PINN:
1. **Training**: The model sees ONLY single-agent data (Vemurafenib-only and Trametinib-only).
2. **Physics**: The model is constrained by pathway ODEs (physics loss) which encode valid interactions.
3. **Testing/Prediction**: The model is evaluated on its ability to predict the **Combination** response (Vem + Tram), which it has NEVER seen.

**Objective**: Validate if the physics-informed inductive bias allows the model to correctly extrapolate the synergistic/combined effects of drugs.

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from train_pinn import train_pinn
from visualize_extrapolation import (
    plot_extrapolation_results, 
    plot_training_history, 
    generate_prediction_table,
    load_pinn_with_data
)
from data_utils import SPECIES_ORDER, VEM_ONLY_DATA, TRAM_ONLY_DATA, VEM_TRAM_DATA
import os

%matplotlib inline

## Step 1: Configure Training Parameters

We train for 10,000 epochs to ensure convergence on the single-agent manifolds. A high physics weight (`5.0`) is critical here: since the model hasn't seen the combination, it must rely on the ODEs to adhere to biological logic when both drugs are present.

In [ ]:
config = {
    'train_until_hour': 48,   # Use all timepoints for single agents
    'num_epochs': 10000,
    'learning_rate': 0.0005,
    'lr_decay': 0.98,
    'batch_size': 12,         # Larger batch to cover both single-agent conditions
    'hidden_size': 128,
    'num_physics_points': 200,
    'weight_decay': 1e-4,
    'weights': {
        'data': 1.0,
        'physics': 5.0,       # High physics weight for extrapolation
        'boundary': 1.0,
        'conservation': 0.5
    }
}

print("Training Configuration:")
print("  Training Data: Vemurafenib Only + Trametinib Only")
print("  Test Data:     Vemurafenib + Trametinib (Combo)")
print(f"  Epochs: {config['num_epochs']}")
print(f"  Physics weight: {config['weights']['physics']}")


## Step 2: Train the Model

The training process will minimize error on single-agent data while enforcing ODE constraints across the entire continuous drug/time space.

In [ ]:
print("="*60)
print("STARTING TRAINING LOOP")
print("="*60)

model, k_params, history, scalers, train_data, test_data = train_pinn(config)


## Step 3: Load & Inspect Model

We load the best model saved during training.

In [ ]:
if os.path.exists('pinn_model_best.pth'):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model, scalers, train_data, test_data = load_pinn_with_data('pinn_model_best.pth', device)
    print("\u2713 Model loaded successfully!")
else:
    print("\u26a0 No trained model found. Run the training cell above first.")


## Step 4: Visualize "Blind" Predictions on Combination Therapy

The plot below compares the model's predictions (lines) against the held-out experimental data for the Combination therapy (red dots).

**Success Criteria:**
- The model should predict the stronger suppression of **pERK** and **pMEK** seen in combination.
- It should capture the rebound or dynamics of **pDGFR** (replacing IGF1R) and **pAKT**.

In [ ]:
plot_extrapolation_results()
plt.show()

## Step 5: Training History

Notice that `Test Loss` here corresponds to the error on the Combination data (which the model never trained on).

In [ ]:
plot_training_history()
plt.show()

# Show final losses
history_df = pd.read_csv('training_history.csv')
final_epoch = history_df.iloc[-1]
print("\nFinal Performance:")
print(f"  Train Loss (Single Agents): {final_epoch['l_data']:.6f}")
print(f"  Test Loss (Combination):    {final_epoch['l_test']:.6f}")


## Step 6: Quantitative Accuracy on Combination Data

We calculate the Percent Error and R² for the prediction of the combination effects.

In [ ]:
predictions_df = generate_prediction_table()

combo_predictions = predictions_df[predictions_df['Dataset'] == 'Test']
if not combo_predictions.empty:
    print("\nCombination Therapy Predictions (Blind Prediction):")
    print(combo_predictions[['Time (hrs)', 'Species', 'True Value', 'Predicted Value', 'Percent Error']].to_string(index=False))

    # Compute R2 per species
    print("\nExtrapolation R² (Predicting Combo from Single Agents):")
    for species in SPECIES_ORDER:
        sub = combo_predictions[combo_predictions['Species'] == species]
        if len(sub) > 1:
            y_true = sub['True Value'].values
            y_pred = sub['Predicted Value'].values
            r2 = 1 - np.sum((y_true - y_pred)**2) / (np.sum((y_true - np.mean(y_true))**2) + 1e-8)
            print(f"  {species}: {r2:.4f}")


## Step 7: Learned Kinetic Parameters

Inspect the kinetic rates learned by fitting the single-agent data.

In [ ]:
checkpoint = torch.load('pinn_model_best.pth', map_location='cpu')
k_params = checkpoint['k_params_state_dict']

print("Learned Kinetic Parameters:")
for name, value in k_params.items():
    print(f"  {name}: {value.item():.4f}")